# CairoGo — Data Types, Duplicates & Integrity Validation

**Owner:** Bassant Tarek (Person C — ID: 20231037)  
**Phase:** Phase 1.3 — Data Cleaning & Validation (`src/data_validation.py` + Section 4 of `06_data_cleaning.ipynb`)  
**Golden Rule:** Reusable transformation and validation functions live in `src/data_validation.py`; this notebook imports those functions and prints shapes, counts, summaries, and `assert` sanity checks.

---
## 1. Setup & Load Raw Datasets
Loads immutable raw copies of `trips.csv`, `customers.csv`, `drivers.csv`, and `zones.csv` into working copies without modifying the original files.

In [8]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_validation import (
    check_duplicate_ids,
    drop_duplicates,
    fix_dtypes,
    validate_numeric_ranges,
    verify_types,
)

PACK_CANDIDATES = [
    ROOT / "data" / "raw" / "CairoGo_pack_v17",
    ROOT / "data" / "raw" / "cairogo_pack",
]
PACK = next((p for p in PACK_CANDIDATES if (p / "trips.csv").exists()), PACK_CANDIDATES[0])

trips_original = pd.read_csv(PACK / "trips.csv")
customers_original = pd.read_csv(PACK / "customers.csv")
drivers_original = pd.read_csv(PACK / "drivers.csv")
zones_original = pd.read_csv(PACK / "zones.csv")

datasets = {
    "trips": trips_original.copy(deep=True),
    "customers": customers_original.copy(deep=True),
    "drivers": drivers_original.copy(deep=True),
    "zones": zones_original.copy(deep=True),
}

print(f"Loaded pack from: {PACK}")
for name, df in datasets.items():
    print(f"{name:10s}: {df.shape[0]:,} rows x {df.shape[1]} columns")

Loaded pack from: C:\Users\H P\cairogo-ride-failure-prediction\data\raw\cairogo_pack
trips     : 27,226 rows x 18 columns
customers : 6,000 rows x 5 columns
drivers   : 1,500 rows x 6 columns
zones     : 24 rows x 5 columns


---
## 2. Raw Schema & Sample Inspection
Inspects initial column data types, unique counts, missing counts, and sample values before standardization.

In [9]:
dtype_report = []
for name, df in datasets.items():
    for column in df.columns:
        dtype_report.append({
            "Dataset": name,
            "Column": column,
            "Current_Type": str(df[column].dtype),
            "Unique_Values": df[column].nunique(),
            "Missing_Values": int(df[column].isnull().sum()),
        })

display(pd.DataFrame(dtype_report))

,Dataset,Column,Current_Type,Unique_Values,Missing_Values
0,trips,trip_id,int64,27226,0
1,trips,customer_id,int64,5350,0
2,trips,driver_id,int64,1494,0
3,trips,request_ts,str,27177,0
4,trips,pickup_zone,str,24,0
5,trips,dropoff_zone,str,24,0
6,trips,distance_km,float64,2902,0
7,trips,surge_multiplier,float64,17,0
8,trips,est_fare,float64,2649,0
9,trips,eta_estimate_min,float64,233,0


In [10]:
for name, df in datasets.items():
    print("\n" + "=" * 60)
    print(f"{name.upper()} SAMPLE VALUES")
    print("=" * 60)
    for column in df.columns:
        print(f"{column}: {df[column].dropna().head(5).tolist()}")


TRIPS SAMPLE VALUES
trip_id: [100001, 100002, 100003, 100004, 100005]
customer_id: [21924, 20391, 24083, 25342, 25697]
driver_id: [5062, 5681, 5407, 5833, 6119]
request_ts: ['2026-05-01 00:01:46', '2026-05-01 00:37:36', '2026-05-01 00:40:12', '2026-05-01 00:49:23', '2026-05-01 01:05:33']
pickup_zone: ['Dokki', 'Dokki', 'Maadi', 'Dokki', 'Dokki']
dropoff_zone: ['Badr City', 'Obour', 'Obour', 'Mohandessin', 'Ramses']
distance_km: [17.22, 12.74, 10.8, 9.51, 4.54]
surge_multiplier: [1.1, 1.1, 1.0, 1.2, 1.0]
est_fare: [107.4, 114.9, 92.3, 75.5, 39.3]
eta_estimate_min: [6.4, 7.3, 6.4, 8.1, 8.6]
weather: ['rain', 'rain', 'rain', 'rain', 'rain']
payment_method: ['card', 'cash', 'card', 'cash', 'cash']
service_level: ['economy', 'comfort', 'comfort', 'economy', 'economy']
request_failed: [0, 0, 0, 0, 0]
actual_trip_time_min: [29.9, 28.9, 22.5, 28.9, 20.6]
cancel_reason: ['driver_rejected', 'no_driver_found', 'no_driver_found', 'driver_cancelled', 'driver_rejected']
support_ticket_flag: [0, 0, 

---
## 3. Data Type Standardization (`fix_dtypes` & `verify_types`)
Applies `src.data_validation.fix_dtypes()` to standardize IDs (`string`), timestamps (`datetime64`), numeric columns (`float64`/`int64`), and low-cardinality categoricals (`category`), then asserts schema compliance with `verify_types()`.

In [11]:
for name, df in datasets.items():
    print("\n" + "=" * 50)
    print(f"{name.upper()} — dtypes before -> after")
    print("=" * 50)
    print("Before:\n", df.dtypes)
    df_fixed = fix_dtypes(df)
    verify_types(df_fixed)
    datasets[name] = df_fixed
    print("\nAfter:\n", df_fixed.dtypes)


TRIPS — dtypes before -> after
Before:
 trip_id                   int64
customer_id               int64
driver_id                 int64
request_ts                  str
pickup_zone                 str
dropoff_zone                str
distance_km             float64
surge_multiplier        float64
est_fare                float64
eta_estimate_min        float64
weather                     str
payment_method              str
service_level               str
request_failed            int64
actual_trip_time_min    float64
cancel_reason               str
support_ticket_flag       int64
driver_arrival_min      float64
dtype: object
All column dtypes verified successfully.

After:
 trip_id                         string
customer_id                     string
driver_id                       string
request_ts              datetime64[us]
pickup_zone                   category
dropoff_zone                  category
distance_km                    float64
surge_multiplier               float64
est_far

---
## 4. Duplicate Rows & Primary Key Validation (`drop_duplicates` & `check_duplicate_ids`)
Removes any exact duplicate rows via `drop_duplicates()` and asserts zero duplicate primary keys via `check_duplicate_ids()`.

In [12]:
primary_keys = {
    "trips": "trip_id",
    "customers": "customer_id",
    "drivers": "driver_id",
    "zones": "zone_id",
}

dup_summary = []
for name, key in primary_keys.items():
    df = datasets[name]
    n_before = len(df)
    df_deduped = drop_duplicates(df)
    n_exact_dups = n_before - len(df_deduped)
    n_pk_dups = check_duplicate_ids(df_deduped, id_col=key, raise_on_duplicates=True)
    datasets[name] = df_deduped

    dup_summary.append({
        "Dataset": name,
        "Primary_Key": key,
        "Total_Rows": len(df_deduped),
        "Exact_Duplicates_Removed": n_exact_dups,
        "Unique_IDs": df_deduped[key].nunique(),
        "Duplicate_ID_Records": n_pk_dups,
    })

display(pd.DataFrame(dup_summary))

Exact duplicate rows removed: 0 (rows: 27226 -> 27226)
Exact duplicate rows removed: 0 (rows: 6000 -> 6000)
Exact duplicate rows removed: 0 (rows: 1500 -> 1500)
Exact duplicate rows removed: 0 (rows: 24 -> 24)


,Dataset,Primary_Key,Total_Rows,Exact_Duplicates_Removed,Unique_IDs,Duplicate_ID_Records
0,trips,trip_id,27226,0,27226,0
1,customers,customer_id,6000,0,6000,0
2,drivers,driver_id,1500,0,1500,0
3,zones,zone_id,24,0,24,0


---
## 5. Referential Integrity & Legacy Zone Audit
Verifies that every `customer_id`, `driver_id`, `pickup_zone`, and `dropoff_zone` in `trips` resolves against `customers`, `drivers`, and `zones` (accounting for `legacy_name` entries handled in Section 2).

In [13]:
trips = datasets["trips"]
customers = datasets["customers"]
drivers = datasets["drivers"]
zones = datasets["zones"]

missing_customer_ids = set(trips["customer_id"].dropna()) - set(customers["customer_id"].dropna())
missing_driver_ids = set(trips["driver_id"].dropna()) - set(drivers["driver_id"].dropna())

valid_zone_names = set(zones["zone_name"].dropna())
valid_zone_or_legacy = valid_zone_names | set(zones["legacy_name"].dropna())

legacy_pickup = set(trips["pickup_zone"].dropna()) - valid_zone_names
legacy_dropoff = set(trips["dropoff_zone"].dropna()) - valid_zone_names
unmatched_pickup = set(trips["pickup_zone"].dropna()) - valid_zone_or_legacy
unmatched_dropoff = set(trips["dropoff_zone"].dropna()) - valid_zone_or_legacy

assert len(missing_customer_ids) == 0, f"Missing customers: {missing_customer_ids}"
assert len(missing_driver_ids) == 0, f"Missing drivers: {missing_driver_ids}"
assert len(unmatched_pickup) == 0, f"Unmatched pickup zones: {unmatched_pickup}"
assert len(unmatched_dropoff) == 0, f"Unmatched dropoff zones: {unmatched_dropoff}"

print("Missing customer IDs:", len(missing_customer_ids))
print("Missing driver IDs  :", len(missing_driver_ids))
print("Legacy pickup zones :", legacy_pickup)
print("Legacy dropoff zones:", legacy_dropoff)
print("Unmatched zones (including legacy): 0")

Missing customer IDs: 0
Missing driver IDs  : 0
Legacy pickup zones : {'Shubra', 'Badr City'}
Legacy dropoff zones: {'Shubra', 'Badr City'}
Unmatched zones (including legacy): 0


---
## 6. Numerical Summary & Domain Range Validation (`validate_numeric_ranges`)
Prints summary statistics and runs `src.data_validation.validate_numeric_ranges()` across `trips` and `drivers`.

In [14]:
numerical_columns = {
    "trips": [
        "distance_km",
        "surge_multiplier",
        "est_fare",
        "eta_estimate_min",
        "actual_trip_time_min",
        "driver_arrival_min",
    ],
    "drivers": [
        "experience_months",
        "rating_avg",
    ],
}

range_reports = []
for name, cols in numerical_columns.items():
    print(f"\n{'=' * 60}")
    print(f"{name.upper()} NUMERICAL SUMMARY")
    print("=" * 60)
    display(datasets[name][cols].describe().T)

    rep = validate_numeric_ranges(datasets[name])
    rep.insert(0, "Dataset", name)
    range_reports.append(rep)

validation_report = pd.concat(range_reports, ignore_index=True)
display(validation_report)
assert (validation_report["Invalid_Values"] == 0).all(), "Numeric range check failed!"


TRIPS NUMERICAL SUMMARY


,count,mean,std,min,25%,50%,75%,max
distance_km,27226.0,10.894424,6.678165,1.14,5.5,9.43,14.620,60.0
surge_multiplier,27226.0,1.257849,0.331323,1.00,1.0,1.10,1.400,2.6
est_fare,27226.0,103.337751,56.722721,27.20,65.4,90.60,124.300,718.0
eta_estimate_min,27226.0,9.015158,4.021844,2.70,5.8,8.20,11.475,29.4
actual_trip_time_min,21560.0,31.912426,12.326115,8.80,22.6,29.30,39.000,101.4
driver_arrival_min,21560.0,9.579875,4.372893,1.00,6.3,9.00,12.400,31.4



DRIVERS NUMERICAL SUMMARY


,count,mean,std,min,25%,50%,75%,max
experience_months,1500.0,38.520000,26.365502,1.00,19.00,32.00,52.00,140.0
rating_avg,1238.0,4.546502,0.184513,3.92,4.42,4.54,4.66,5.0


,Dataset,Column,Invalid_Values,Status
0,trips,distance_km,0,Passed
1,trips,surge_multiplier,0,Passed
2,trips,est_fare,0,Passed
3,trips,eta_estimate_min,0,Passed
4,trips,actual_trip_time_min,0,Passed
5,trips,driver_arrival_min,0,Passed
6,drivers,experience_months,0,Passed
7,drivers,rating_avg,0,Passed
